# followup_report

> Summaries, `report.md` and figures for the follow-up runs (speedup plan Parts B and D): the hard board against difficulty 8, model size at low and matched medium effort, rounds and soft-territory squads (numpy and LLM), and the team-total curve.

In [ ]:
#| default_exp followup_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.llm_figures import COLORS, _legend, _save, figTeamTotal
from explRL.llm_report import NAN, _f, _fmt, _paired, _sel, _stat, interfaceTable

HAIKU, SONNET = 'claude-haiku-5-5', 'claude-sonnet-5-5'


def _cells(rows, keys, metrics):
    out = {}
    for r in rows:
        out.setdefault(tuple(int(_f(r[k])) if k == 'N' else r[k] for k in keys), []).append(r)
    return {k: {m: _stat(_f(r.get(m, NAN)) for r in v) for m in metrics} for k, v in sorted(out.items())}


def hardTable(rows, p10):
    "D1: cells of the hard board; blackboard minus none at N = 4 there and at difficulty 8 (Phase 10 main)."
    hard = _sel(rows, study='hard')
    cells = _cells(hard, ('channel', 'N'), ('welfare', 'teamWelfare', 'solved', 'opens', 'wrong', 'duplicates', 'maxTokenStops'))
    c = lambda rs, s, m: _stat(_paired(_sel(rs, study=s, task='board', channel='blackboard', N=4),
                                       _sel(rs, study=s, task='board', channel='none', N=4), m))
    return dict(cells=cells, level=sorted({r['level'] for r in hard}),
                contrast={m: c(hard, 'hard', m) for m in ('welfare', 'teamWelfare')},
                phase10={m: c(p10, 'main', m) for m in ('welfare', 'teamWelfare')})


def _benefit(rs, study, model, effort):
    "{seed: blackboard welfare minus none welfare} at N = 4 on the board for one model and effort."
    sel = lambda ch: {int(_f(r['seed'])): _f(r['welfare']) for r in _sel(rs, study=study, task='board', channel=ch, N=4,
                                                                           model=model) if r['effort'] == effort}
    a, b = sel('blackboard'), sel('none')
    return {k: a[k] - b[k] for k in sorted(set(a) & set(b))}


def effortTable(rows, p10):
    "D2: blackboard benefit at N = 4 by model and effort; Sonnet minus Haiku, paired on seeds 500-502."
    seeds = {500, 501, 502}
    b = {('medium', HAIKU): _benefit(rows, 'effort', HAIKU, 'medium'), ('medium', SONNET): _benefit(rows, 'effort', SONNET, 'medium'),
         ('low', HAIKU): _benefit(p10, 'main', HAIKU, 'low'), ('low', SONNET): _benefit(p10, 'sonnet', SONNET, 'low')}
    out = {f'benefit {e} {m.split("-")[1]}': _stat(v[s] for s in sorted(seeds & set(v))) for (e, m), v in b.items()}
    for e in ('low', 'medium'):
        s, h = b[(e, SONNET)], b[(e, HAIKU)]
        out[f'Sonnet minus Haiku, effort {e}'] = _stat(s[k] - h[k] for k in sorted(seeds & set(s) & set(h)))
    per = {}
    for r in _sel(rows, study='effort') + [r for r in p10 if r['study'] in ('main', 'sonnet') and r['task'] == 'board'
                                           and int(_f(r['seed'])) in seeds and int(_f(r['N'])) in (1, 4)]:
        per.setdefault((r['effort'], r['model'].split('-')[1]), []).append(r)
    out['perRun'] = {k: dict(wrong=_stat(_f(r['wrong']) for r in v),
                             outputPerCall=sum(_f(r['outputTokens']) for r in v) / max(sum(_f(r['calls']) for r in v), 1))
                     for k, v in sorted(per.items())}
    return out


def roundsNumpyTable(rows):
    "B-numpy: per N and arm, perAgentSecond and team total, and each arm minus events (paired on seed)."
    rs = _sel(rows, study='roundsNumpy')
    cells = _cells(rs, ('N', 'level'), ('perAgentSecond', 'teamWelfare', 'homeShare', 'coverage', 'diversity'))
    diff = {(N, lv): _stat(_paired(_sel(rs, N=N, level=lv), _sel(rs, N=N, level='events'), 'perAgentSecond'))
            for (N, lv) in cells if lv != 'events'}
    return dict(cells=cells, diff=diff)


def roundsLLMTable(rows, p10):
    "B-LLM at N = 8: events (Phase 10 main cells, seeds 500-502), rounds, squads; differences paired on seed."
    ev = [r for r in _sel(p10, study='main', task='forage', channel='blackboard', N=8) if int(_f(r['seed'])) in (500, 501, 502)]
    arms = dict(events=ev, **{lv: _sel(rows, study='rounds', level=lv) for lv in sorted({r['level'] for r in _sel(rows, study='rounds')})})
    ms = ('welfare', 'teamWelfare', 'exploreShare', 'diversity', 'homeShare')
    out = dict(cells={a: {m: _stat(_f(r.get(m, NAN)) for r in v) for m in ms} for a, v in arms.items()})
    sq = [a for a in arms if a.startswith('squads')]
    out['diff'] = {f'rounds - events: {m}': _stat(_paired(arms.get('rounds', []), ev, m)) for m in ms[:4]}
    out['diff'].update({f'{a} - rounds: {m}': _stat(_paired(arms[a], arms.get('rounds', []), m)) for a in sq for m in ms[:4]})
    return out


def summaries(rows, p10):
    return dict(hard=hardTable(rows, p10), effort=effortTable(rows, p10), numpy=roundsNumpyTable(rows),
                llm=roundsLLMTable(rows, p10), interface=interfaceTable(rows))


def writeReport(path, S, meta):
    "report.md: provenance first, then D1, D2, B-numpy, B-LLM and the interface table (all exploratory)."
    L = [f"# Follow-up runs after Phase 10 ({meta['mode']})", '', 'Exploratory, pre-specified in `FOLLOWUP_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; models {', '.join(meta['models']) or 'none'}",
         f"- Prices checked {meta['pricesChecked']}; cost of these runs ${meta['cost']:.2f}; total spend ${meta['spent']:.2f} "
         f"of the ${meta['cap']:.0f} cap; calls {meta['calls']}; wall time {meta['wall']:.0f} s; retries {meta.get('retries', 0)}", '',
         f"## D1 hard board ({', '.join(S['hard']['level']) or 'not run'})", '', '| channel | N | welfare | team total | solved | opened | wrong | max_tokens |',
         '|---|---|---|---|---|---|---|---|']
    L += [f"| {ch} | {N} | {_fmt(v['welfare'])} | {_fmt(v['teamWelfare'])} | {v['solved']['mean']:.1f} | {v['opens']['mean']:.1f} | "
          f"{v['wrong']['mean']:.1f} | {v['maxTokenStops']['mean']:.1f} |" for (ch, N), v in S['hard']['cells'].items()]
    L += ['', '| blackboard minus none, N = 4 | hard board | difficulty 8 (Phase 10) |', '|---|---|---|']
    L += [f"| {m} | {_fmt(S['hard']['contrast'][m])} | {_fmt(S['hard']['phase10'][m])} |" for m in ('welfare', 'teamWelfare')]
    L += ['', '## D2 model size at matched effort (board, N = 4)', '', '| contrast | mean [95% CI] |', '|---|---|']
    L += [f'| {k} | {_fmt(v)} |' for k, v in S['effort'].items() if k != 'perRun']
    L += ['', '| effort | model | wrong answers per run | output tokens per call |', '|---|---|---|---|']
    L += [f"| {e} | {m} | {v['wrong']['mean']:.1f} | {v['outputPerCall']:.0f} |" for (e, m), v in S['effort']['perRun'].items()]
    L += ['', '## B-numpy: rounds and squads (perAgentSecond; arm minus events)', '', '| N | arm | perAgentSecond | minus events | home share |', '|---|---|---|---|---|']
    L += [f"| {N} | {lv} | {_fmt(v['perAgentSecond'])} | {_fmt(S['numpy']['diff'][(N, lv)]) if lv != 'events' else ''} | "
          f"{v['homeShare']['mean']:.2f} |" for (N, lv), v in S['numpy']['cells'].items()]
    L += ['', '## B-LLM: rounds and squads (text foraging, blackboard, N = 8)', '', '| arm | welfare | team total | explore share | home share |', '|---|---|---|---|---|']
    L += [f"| {a} | {_fmt(v['welfare'])} | {_fmt(v['teamWelfare'])} | {_fmt(v['exploreShare'])} | {v['homeShare']['mean']:.2f} |"
          for a, v in S['llm']['cells'].items()]
    L += ['', '| difference | mean [95% CI] |', '|---|---|'] + [f'| {k} | {_fmt(v)} |' for k, v in S['llm']['diff'].items()]
    if meta.get('runWall'):
        L += ['', '## Wall time per run (s, mean by cell level)', '', '| level | seconds |', '|---|---|']
        L += [f'| {lv} | {sec:.0f} |' for lv, sec in sorted(meta['runWall'].items())]
    path.write_text('\n'.join(L) + '\n')
    return path


def figBars(items, path, title, ylabel):
    "Bar chart of (label, stats) items with 95% CIs."
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(items) + 2), 3.6))
    m = np.array([s['mean'] for _, s in items])
    err = [m - np.array([s['ci'][0] for _, s in items]), np.array([s['ci'][1] for _, s in items]) - m]
    ax.bar(range(len(items)), m, yerr=err, capsize=3, color='tab:blue')
    ax.set_xticks(range(len(items)), [k for k, _ in items], fontsize=7, rotation=20, ha='right')
    ax.axhline(0, color='k', lw=0.5)
    ax.set(title=title, ylabel=ylabel)
    return _save(fig, path)


def figRounds(S, path):
    "B-numpy: perAgentSecond by arm for each N."
    fig, ax = plt.subplots(figsize=(8, 3.8))
    cells = S['numpy']['cells']
    arms = sorted({lv for _, lv in cells}, key=lambda a: (a != 'events', a != 'rounds', a))
    for N in sorted({N for N, _ in cells}):
        ax.plot(range(len(arms)), [cells.get((N, a), {}).get('perAgentSecond', dict(mean=NAN))['mean'] for a in arms],
                marker='o', label=f'N = {N}')
    ax.set_xticks(range(len(arms)), arms, fontsize=7, rotation=20, ha='right')
    ax.set(ylabel='welfare per agent-second', title='Rounds and soft-territory squads (numpy, blackboard)')
    _legend(ax)
    return _save(fig, path)


def writeFigures(out, rows, p10, S):
    return [figBars(list(S['hard']['contrast'].items()) + [(f'{k} (d = 8)', v) for k, v in S['hard']['phase10'].items()],
                    out / 'hard.png', 'Blackboard benefit at N = 4: hard board vs difficulty 8', 'blackboard minus none'),
            figBars([(k, v) for k, v in S['effort'].items() if k != 'perRun'], out / 'effort.png',
                    'Model size at low and medium effort (board, N = 4)', 'welfare per agent-turn'),
            figRounds(S, out / 'rounds.png'), figTeamTotal(p10, out / 'team_total.png')]

## Tests

In [ ]:
import tempfile
from pathlib import Path
from explRL.followup_report import *
from explRL.followup_grid import followupQuick, effortGrid, hardGrid, roundsGrid
from explRL.llm_grid import quickGrid
from explRL.llm_sched import runWorkload

cells = followupQuick() + [c for c in hardGrid(10, 8000) + effortGrid() + roundsGrid(1.0) if c['seed'] in (500, 643)]
for c in cells:
    c['cfg']['turns'] = 3
rows = runWorkload([('f', cells)], dict(kind='fake'), 8)['rows']
p10 = runWorkload([('q', [c for c in quickGrid() if c['study'] in ('main', 'sonnet')])], dict(kind='fake'), 8)['rows']
S = summaries(rows, p10)
assert set(S['hard']['cells']) == {('blackboard', 1), ('blackboard', 4), ('none', 1), ('none', 4)} and S['hard']['contrast']['welfare']['n'] == 1
assert S['hard']['phase10']['welfare']['n'] == 1 and S['effort']['Sonnet minus Haiku, effort medium']['n'] == 1
assert S['effort']['Sonnet minus Haiku, effort low']['n'] == 1 and ('medium', 'sonnet') in S['effort']['perRun']
assert (8, 'squads|beta=inf') in S['numpy']['cells'] and (8, 'events') not in S['numpy']['diff']
assert set(S['llm']['cells']) == {'events', 'rounds', 'squads|beta=1'} and S['llm']['diff']['squads|beta=1 - rounds: welfare']['n'] == 1
meta = dict(mode='quick', planHash='abc123', commit='c0ffee', models=['claude-haiku-5-5'], pricesChecked='2026-10-12', cost=0.12, runWall={'rounds': 12.0},
            spent=33.5, cap=150.0, calls=100, wall=2.0, retries=0)
out = Path(tempfile.mkdtemp())
txt = writeReport(out / 'report.md', S, meta).read_text()
for s in ('abc123', 'c0ffee', '$33.50', '| rounds | 12 |', 'D1 hard board', 'D2 model size', 'B-numpy', 'B-LLM', 'Sonnet minus Haiku, effort medium'):
    assert s in txt, s
assert [p.name for p in writeFigures(out, rows, p10, S)] == ['hard.png', 'effort.png', 'rounds.png', 'team_total.png']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()